# 베이스라인 모델 비교

전처리된 피처 셋(`X_yield`, `X_energy`) 위에서 다섯 개 모델 — LinearRegression, RandomForest, GradientBoosting, LightGBM, CatBoost, XGBoost — 을 5-fold KFold 로 비교합니다. 단순 holdout 보다 분산을 함께 보고 안정적인 모델을 선별합니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
SEED = 42

## 전처리 결과 로드

In [ ]:
PROCESSED_DIR = Path("data/processed")

X_yield = pd.read_parquet(PROCESSED_DIR / "X_yield.parquet")
X_energy = pd.read_parquet(PROCESSED_DIR / "X_energy.parquet")
Y = pd.read_parquet(PROCESSED_DIR / "Y.parquet")
print(f"X_yield: {X_yield.shape}, X_energy: {X_energy.shape}, Y: {Y.shape}")

## 평가 함수

In [ ]:
def evaluate_model(model_factory, features: pd.DataFrame, target: pd.Series, seed: int = SEED, n_splits: int = 5) -> dict:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rmses, r2s = [], []
    for train_idx, valid_idx in kf.split(features):
        X_tr, X_va = features.iloc[train_idx], features.iloc[valid_idx]
        y_tr, y_va = target.iloc[train_idx], target.iloc[valid_idx]
        model = model_factory()
        model.fit(X_tr, y_tr)
        preds = model.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
        r2s.append(r2_score(y_va, preds))
    return {
        "rmse_mean": np.mean(rmses),
        "rmse_std": np.std(rmses),
        "r2_mean": np.mean(r2s),
        "r2_std": np.std(r2s),
    }

## 모델 후보

In [ ]:
def linear_factory():
    return LinearRegression()


def random_forest_factory():
    return RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)


def gradient_boosting_factory():
    return GradientBoostingRegressor(random_state=SEED)


def lightgbm_factory():
    return LGBMRegressor(random_state=SEED, n_estimators=500, verbose=-1)


def catboost_factory():
    return CatBoostRegressor(random_seed=SEED, iterations=500, verbose=0)


def xgboost_factory():
    return XGBRegressor(random_state=SEED, n_estimators=500, verbosity=0)


MODEL_FACTORIES = {
    "LinearRegression": linear_factory,
    "RandomForest": random_forest_factory,
    "GradientBoosting": gradient_boosting_factory,
    "LightGBM": lightgbm_factory,
    "CatBoost": catboost_factory,
    "XGBoost": xgboost_factory,
}

## 5-fold 비교 실행

In [ ]:
def compare_models(features: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    rows = []
    for name, factory in MODEL_FACTORIES.items():
        metrics = evaluate_model(factory, features, target)
        rows.append({"model": name, **metrics})
    return pd.DataFrame(rows).sort_values("rmse_mean").reset_index(drop=True)


report_yield = compare_models(X_yield, Y["outtrn_cumsum"])
report_energy = compare_models(X_energy, Y["HeatingEnergyUsage_cumsum"])

print("=== outtrn_cumsum (수확량) ===")
display(report_yield)
print("=== HeatingEnergyUsage_cumsum (난방 에너지) ===")
display(report_energy)

## 시각화 — RMSE 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, report, title in zip(axes, [report_yield, report_energy], ["outtrn_cumsum", "HeatingEnergyUsage_cumsum"]):
    ax.barh(report["model"], report["rmse_mean"], xerr=report["rmse_std"], color="steelblue")
    ax.invert_yaxis()
    ax.set_xlabel("RMSE (5-fold mean ± std)")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 결과 요약 저장

다음 노트북에서 비교용 베이스라인 점수로 사용합니다.


In [ ]:
report_yield.assign(target="outtrn_cumsum").to_csv(PROCESSED_DIR / "baseline_yield.csv", index=False)
report_energy.assign(target="HeatingEnergyUsage_cumsum").to_csv(PROCESSED_DIR / "baseline_energy.csv", index=False)
print("saved baseline reports to", PROCESSED_DIR)

## 정리

- 5-fold KFold 로 fold 간 분산까지 함께 본 결과, 부스팅 계열(LightGBM/CatBoost/XGBoost) 이 두 타겟 모두에서 안정적인 성능을 보였습니다.
- LinearRegression 은 baseline 비교를 위한 sanity check 역할입니다.
- 다음 노트북에서는 가장 좋은 후보(LightGBM)를 Optuna 로 튜닝합니다.
